# Переразметка манифеста

Меняет метки в готовом манифесте, не перечитывая кадры.

Метрики — резкость, яркость, перцептивный хеш, размеры — от правил разметки не
зависят и уже посчитаны. Меняется единственная колонка `labels`, а это
преобразование таблицы на несколько сотен килобайт: секунды вместо повторного
чтения гигабайтов картинок и без всякой отдельной машины.

Ровно ради этого манифест и был сделан первичным артефактом.

## Повод

Метки CADC оказались неверными, проверено по кадрам в `samples.ipynb`:

* заезды 2018 года (`lens_snow/`) — голый мокрый асфальт, снега нет, на стекле
  талая вода. Было `snowfall` + `soiling`, должно быть `raindrops`;
* заезды 2019 года (`clear_lens/`) — настоящий снегопад. Остаётся `snowfall`.

Разбор — в `data/INSIGHTS.md`. Правила-источник истины — в `data/sources.yaml`.

## 1. Окружение

In [ ]:
%pip install -q boto3 pyarrow pandas

import io, os, socket, datetime
import boto3, pandas as pd, pyarrow as pa, pyarrow.parquet as pq

BUCKET   = os.environ.get("DATASETS_BUCKET", "occlusionnet-clearml-b052c3-datasets")
ENDPOINT = "https://storage.yandexcloud.net"
REGION   = "ru-central1"

missing = [k for k in ("S3_KEY", "S3_SECRET") if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"нет ключей {missing}: секреты проекта DataSphere "
                       "и перезапуск ядра, либо .env для локального ядра")

s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name=REGION,
                  aws_access_key_id=os.environ["S3_KEY"],
                  aws_secret_access_key=os.environ["S3_SECRET"])
print("хост:", socket.gethostname(), "| бакет:", BUCKET)

## 2. Что переразмечаем

Правила ниже обязаны совпадать с секцией `labels` соответствующего источника в
`data/sources.yaml`. Держать их в двух местах плохо, но у удалённого ядра
репозитория нет; расхождение видно по распределению меток в конце.

In [ ]:
SOURCE = "cadc"
KEY    = f"manifest/{SOURCE}.parquet"

# sources.yaml -> cadc.labels.from_path.lens
RULES = {
    "lens_snow":  ["raindrops"],   # талая вода на стекле, снега в кадре нет
    "clear_lens": ["snowfall"],    # снегопад; часть кадров ещё и с каплями
}

def new_labels(raw_path):
    group = raw_path.split("/")[0]
    if group not in RULES:
        raise KeyError(f"нет правила для группы {group!r}")
    return sorted(RULES[group])

## 3. Читаем и смотрим, что было

In [ ]:
body = s3.get_object(Bucket=BUCKET, Key=KEY)["Body"].read()
table = pq.read_table(io.BytesIO(body))
print(f"{KEY}: {table.num_rows:,} строк, {len(body)/1024:.0f} КБ")

df = table.to_pandas()
before = df.labels.apply(lambda l: "+".join(sorted(l)) or "clean").value_counts()
print("\nбыло:")
print(before.to_string())

## 4. Подменяем колонку

Не через `pandas.to_parquet`: тот выведет типы заново и может испортить схему —
`phash` там uint64, `labels` это список строк, `mask_nonempty` допускает null.
Меняем одну колонку в таблице, остальные остаются побайтово теми же.

In [ ]:
labels_new = [new_labels(p) for p in df.raw_path]

idx = table.schema.get_field_index("labels")
table_new = table.set_column(
    idx,
    table.schema.field(idx),                       # то же имя и тот же тип
    pa.array(labels_new, type=pa.list_(pa.string())),
)

after = pd.Series(["+".join(l) or "clean" for l in labels_new]).value_counts()
print("стало:")
print(after.to_string())

changed = sum(sorted(a) != sorted(b) for a, b in zip(df.labels, labels_new))
print(f"\nизменено строк: {changed:,} из {len(df):,}")
print("схема совпадает:", table_new.schema.equals(table.schema))

## 5. Записываем, сохранив старую версию

Перезапись без копии — потеря работы: восстановить манифест можно только
повторным чтением всех кадров. Старая версия уезжает в `manifest/archive/`
с отметкой времени.

In [ ]:
stamp  = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
backup = f"manifest/archive/{SOURCE}-{stamp}.parquet"

s3.put_object(Bucket=BUCKET, Key=backup, Body=body)
print("старая версия сохранена:", backup)

buf = io.BytesIO()
pq.write_table(table_new, buf, compression="zstd")
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key=KEY, Body=buf.getvalue())
print(f"записан {KEY}: {buf.getbuffer().nbytes/1024:.0f} КБ")

## 6. Проверка: читаем обратно из бакета

In [ ]:
check = pq.read_table(io.BytesIO(
    s3.get_object(Bucket=BUCKET, Key=KEY)["Body"].read())).to_pandas()

print("строк:", f"{len(check):,}")
print(check.labels.apply(lambda l: "+".join(sorted(l)) or "clean")
           .value_counts().to_string())

# метрики обязаны остаться нетронутыми
same = all(check[c].equals(df[c]) for c in
           ("frame_uid", "raw_path", "sharpness", "phash", "width", "height"))
print("\nметрики и пути не изменились:", same)

## Что дальше

После переразметки стоит перезапустить `samples.ipynb` и посмотреть на
`raindrops` и `snowfall` заново — теперь картинки должны соответствовать
подписям.

Отдельный вопрос, оставшийся открытым: у `clear_lens` на части кадров тоже есть
капли, а аннотация CADC даётся на заезд целиком. Честные пер-кадровые метки для
этих 875 кадров можно получить только просмотром глазами — это полчаса работы и
даст заодно небольшой размеченный вручную набор для валидации.